In [3]:
# =========================
# DAF: Context-Aware CIS
# (robust single-cell, includes dataset counts)
# =========================

from __future__ import annotations
import os
from collections import Counter
from typing import Dict, Iterable, List, Optional, Tuple, Any
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------
# Config: dataset -> profile
# -------------------------
DATASET_PURPOSE = {
    "NSL-KDD": "general_enterprise_nids",
    "UNSW-NB15": "general_enterprise_nids",
    "CICIDS2017": "general_enterprise_nids",
    "CSE-CIC-IDS2018": "general_enterprise_nids",
    "ToN_IoT": "iot_edge",
    "BoT_IoT": "iot_edge",
    "HIKARI-2021": "encrypted_flow_only_ids",
    "UNR-IDD": "sdn_switch_telemetry_ids",
}

PROFILE_PRIORS = {
    "general_enterprise_nids": {
        "Benign": 0.985,
        "Probe": 0.005, "DoS": 0.003, "BruteForce": 0.002,
        "WebAttack": 0.002, "BotMalware": 0.002, "CryptoMining": 0.001,
        "Other": 0.000
    },
    "iot_edge": {
        "Benign": 0.990,
        "Probe": 0.004, "BruteForce": 0.002, "DoS": 0.002,
        "CryptoMining": 0.001, "BotMalware": 0.0008, "WebAttack": 0.0008, "Other": 0.0004
    },
    "sdn_switch_telemetry_ids": {
        "Benign": 0.9950,
        "Port scan": 0.0012,
        "TCP-SYN flood": 0.0015,
        "Flow table overflow": 0.0010,
        "Blackhole": 0.0007,
        "Diversion": 0.0006,
        "Other": 0.0000
    },
    "encrypted_flow_only_ids": "general_enterprise_nids"
}

PROFILE_ALIASES = {
    "sdn_switch_telemetry_ids": {"Probe": "Port scan", "DoS": "TCP-SYN flood"}
}

CANONICAL_MAP = {
    # benign
    "BENIGN": "Benign", "Benign": "Benign", "benign": "Benign",
    "Background": "Benign", "Normal": "Benign", "normal": "Benign",

    # NSL-KDD DoS
    "neptune": "DoS", "smurf": "DoS", "teardrop": "DoS", "pod": "DoS",
    "land": "DoS", "back": "DoS", "apache2": "DoS", "processtable": "DoS",
    "mailbomb": "DoS", "udpstorm": "DoS",

    # NSL-KDD Probe
    "satan": "Probe", "ipsweep": "Probe", "nmap": "Probe",
    "portsweep": "Probe", "mscan": "Probe", "saint": "Probe",

    # brute-force
    "guess_passwd": "BruteForce", "snmpguess": "BruteForce",
    "snmpgetattack": "BruteForce",

    # web/other
    "phf": "WebAttack", "sqlattack": "WebAttack",
    "httptunnel": "Other", "buffer_overflow": "Other",
    "loadmodule": "Other", "rootkit": "Other", "perl": "Other",
    "ps": "Other", "sendmail": "Other", "xterm": "Other",
    "imap": "Other", "multihop": "Other", "ftp_write": "Other",
    "warezmaster": "Other", "warezclient": "Other",
    "xlock": "Other", "xsnoop": "Other", "spy": "Other",
    "named": "Other",

    # UNSW & generic
    "Reconnaissance": "Probe", "DoS": "DoS",
    "Worms": "BotMalware", "Backdoor": "BotMalware",
    "Generic": "Other", "Exploits": "Other", "Fuzzers": "Other",
    "Analysis": "Other", "Shellcode": "Other",

    # CIC etc
    "PortScan": "Probe",
    "DoS Hulk": "DoS", "DoS GoldenEye": "DoS",
    "DoS slowloris": "DoS", "DoS Slowhttptest": "DoS", "DDoS": "DoS",
    "FTP-Patator": "BruteForce", "SSH-Patator": "BruteForce",
    "Web Attack   Brute Force": "WebAttack",
    "Web Attack   XSS": "WebAttack",
    "Web Attack   Sql Injection": "WebAttack",
    "Bot": "BotMalware", "Infiltration": "Other", "Heartbleed": "Other",

    # IoT/other
    "scanning": "Probe", "ddos": "DoS", "dos": "DoS",
    "injection": "WebAttack", "xss": "WebAttack",
    "password": "BruteForce", "backdoor": "BotMalware", "ransomware": "BotMalware",
    "mitm": "Other", "Reconnaissance": "Probe", "Theft": "Other",

    # HIKARI
    "Probing": "Probe",
    "Bruteforce": "BruteForce", "Bruteforce-XML": "BruteForce",
    "XMRIGCC CryptoMiner": "CryptoMining",

    # UNR-IDD
    "TCP-SYN": "TCP-SYN flood", "PortScan": "Port scan",
    "Overflow": "Flow table overflow", "Blackhole": "Blackhole", "Diversion": "Diversion",
}

BENIGN_LABELS = {"BENIGN", "Benign", "benign", "Background", "Normal", "normal"}

# -------------------------
# Core helpers (math)
# -------------------------
def _norm_hhi(probs: np.ndarray) -> float:
    probs = np.asarray(probs, dtype=float)
    probs = probs[probs > 0]
    K = probs.size
    if K == 0:
        return 0.0
    if K == 1:
        return 1.0
    S = float(np.sum(probs ** 2))
    return (S - 1.0 / K) / (1.0 - 1.0 / K)

def _js_div(p: np.ndarray, q: np.ndarray) -> float:
    p = np.asarray(p, dtype=float); q = np.asarray(q, dtype=float)
    if p.sum() == 0 or q.sum() == 0:
        return 0.0
    p = p / p.sum(); q = q / q.sum()
    m = 0.5 * (p + q)
    def _kl(a, b):
        mask = (a > 0) & (b > 0)
        if not mask.any(): return 0.0
        return float(np.sum(a[mask] * (np.log2(a[mask]) - np.log2(b[mask]))))
    js = 0.5 * _kl(p, m) + 0.5 * _kl(q, m)
    return float(max(0.0, min(1.0, js)))

def _resolve_profile(profile_name: str) -> str:
    val = PROFILE_PRIORS.get(profile_name)
    return val if isinstance(val, str) else profile_name

def _build_profile_prior(classes: List[str], dataset_name: str) -> np.ndarray:
    profile = DATASET_PURPOSE.get(dataset_name)
    if profile is None:
        return np.ones(len(classes), dtype=float) / float(len(classes))
    profile = _resolve_profile(profile)
    prof = PROFILE_PRIORS.get(profile, {})
    alias = PROFILE_ALIASES.get(profile, {})
    q = []
    for c in classes:
        bucket = CANONICAL_MAP.get(c, "Other")
        bucket = alias.get(bucket, bucket)
        w = prof.get(bucket, prof.get("Other", 0.0))
        q.append(w)
    q = np.array(q, dtype=float)
    if q.sum() <= 0:
        return np.ones_like(q) / float(len(q))
    return q / q.sum()

# -------------------------
# Robust input coercion
# -------------------------
def _coerce_counts(counts: Any) -> Dict[str, int]:
    """
    Accept:
      - dict[str,int] -> returned as-is
      - iterable of (label,count) pairs -> dict(created)
      - iterable of raw labels (str) -> Counter(list) -> dict
    Raises ValueError with clear message if input looks like a placeholder or unsupported type.
    """
    if isinstance(counts, dict):
        # quick sanity: check for Ellipsis or placeholder
        if any(k is ... or v is ... for k, v in counts.items()):
            raise ValueError("Counts dict contains Ellipsis '...' placeholder; replace with real counts.")
        return counts
    # If it's an iterable (list, set, tuple, generator)
    try:
        # first try pairwise (label,count) mapping
        # Convert to list to safely iterate multiple times
        items = list(counts)
    except TypeError:
        raise ValueError("Provided counts is not iterable. Please provide a dict[label->count], "
                         "an iterable of (label,count) pairs, or an iterable of labels.")
    # Detect obvious placeholder like {...} -> results in set containing Ellipsis or other non-str values
    if len(items) == 1 and items[0] is ...:
        raise ValueError("Detected placeholder '...'. Replace with the actual class-count dictionary.")
    # If items are pairs (len==2 and second element numeric)
    if all(isinstance(it, (list, tuple)) and len(it) == 2 for it in items):
        d = dict(items)
        return d
    # If items are strings (labels), treat as label list and count frequencies
    if all(isinstance(it, str) for it in items):
        return dict(Counter(items))
    # If items look like singletons not strings, raise a helpful error
    raise ValueError("Unable to coerce 'counts' into a dict. Please pass either:\n"
                     "  - a dict[label->count], or\n"
                     "  - an iterable of (label,count) pairs, or\n"
                     "  - an iterable of raw labels (str).")

# -------------------------
# Main computation
# -------------------------
def compute_cis_from_counts(counts_input: Any, dataset_name: str, beta: float = 0.20) -> Dict[str, float]:
    counts = _coerce_counts(counts_input)
    classes = list(counts.keys())
    total = float(sum(counts.values()))
    if total <= 0:
        raise ValueError("Empty counts (sum zero). Provide non-empty class counts.")
    p = np.array([counts[c] / total for c in classes], dtype=float)
    K = len(classes)

    CIS_all = _norm_hhi(p)
    benign_mask = np.array([c in BENIGN_LABELS for c in classes], dtype=bool)

    p_att = p.copy(); p_att[benign_mask] = 0.0
    if p_att.sum() > 0:
        p_att = p_att / p_att.sum()
    nonzero_att = int((p_att > 0).sum())
    CIS_attack = _norm_hhi(p_att[p_att > 0]) if nonzero_att > 1 else (1.0 if nonzero_att == 1 else 0.0)

    p_benign = float(p[benign_mask].sum())
    ABI = (p_benign - (1.0 / K)) / (1.0 - (1.0 / K)) if K > 1 else 1.0

    CIS_base = 0.5 * CIS_all + 0.3 * CIS_attack + 0.2 * ABI

    q = _build_profile_prior(classes, dataset_name)
    JS_to_profile = _js_div(p, q)

    CIS_CA = (1.0 - beta) * CIS_base + beta * JS_to_profile
    CBA_CA = 1.0 - CIS_CA

    return {
        "dataset": dataset_name,
        "K": int(K),
        "p_benign": float(p_benign),
        "CIS_all": float(CIS_all),
        "CIS_attack": float(CIS_attack),
        "ABI": float(ABI),
        "CIS_base": float(CIS_base),
        "JS_to_profile": float(JS_to_profile),
        "CIS_CA": float(CIS_CA),
        "CBA_CA": float(CBA_CA),
    }

# -------------------------
# Datasets (label counts)
# -------------------------
NSL_KDD = {
    "normal": 77053, "neptune": 45870, "satan": 4368, "ipsweep": 3740, "smurf": 3311, "portsweep": 3088,
    "nmap": 1566, "back": 1315, "guess_passwd": 1284, "mscan": 996, "warezmaster": 964, "teardrop": 904,
    "warezclient": 890, "apache2": 737, "processtable": 685, "snmpguess": 331, "saint": 319, "mailbomb": 293,
    "pod": 242, "snmpgetattack": 178, "httptunnel": 133, "buffer_overflow": 50, "land": 25, "multihop": 25,
    "rootkit": 23, "named": 17, "ps": 15, "sendmail": 14, "xterm": 13, "imap": 12, "loadmodule": 11,
    "ftp_write": 11, "xlock": 9, "phf": 6, "perl": 5, "xsnoop": 4, "spy": 2, "worm": 2, "sqlattack": 2,
    "udpstorm": 2
}

UNSW_NB15 = {
    "Normal": 93000, "Generic": 58871, "Exploits": 44525, "Fuzzers": 24246, "DoS": 16353,
    "Reconnaissance": 13987, "Analysis": 2677, "Backdoor": 2329, "Shellcode": 1511, "Worms": 174
}

CICIDS2017 = {
    "BENIGN": 2273097, "DoS Hulk": 231073, "PortScan": 158930, "DDoS": 128027, "DoS GoldenEye": 10293,
    "FTP-Patator": 7938, "SSH-Patator": 5897, "DoS slowloris": 5796, "DoS Slowhttptest": 5499, "Bot": 1966,
    "Web Attack   Brute Force": 1507, "Web Attack   XSS": 652, "Infiltration": 36,
    "Web Attack   Sql Injection": 21, "Heartbleed": 11
}

CSE_CSC_IDS2018 = {
    "Benign": 13484708, "DDOS attack-HOIC": 686012, "DDoS attacks-LOIC-HTTP": 576191, "DoS attacks-Hulk": 461912,
    "Bot": 286191, "FTP-BruteForce": 193360, "SSH-Bruteforce": 187589, "Infilteration": 161934,
    "DoS attacks-SlowHTTPTest": 139890, "DoS attacks-GoldenEye": 41508, "DoS attacks-Slowloris": 10990,
    "DDOS attack-LOIC-UDP": 1730, "Brute Force -Web": 611, "Brute Force -XSS": 230, "SQL Injection": 87
}

ToN_IOT = {
    "normal": 41877, "scanning": 20000, "ddos": 19986, "injection": 19952, "password": 19791,
    "dos": 17984, "backdoor": 17488, "ransomware": 13226, "xss": 11961, "mitm": 1039
}

BoT_IoT = {
    "Normal": 477, "DDoS": 1926624, "DoS": 1650260, "Reconnaissance": 91082, "Theft": 79
}

HIKARI_2021 = {
    "Benign": 347431, "Background": 170151, "Probing": 23388,
    "Bruteforce": 5884, "Bruteforce-XML": 5145, "XMRIGCC CryptoMiner": 3279
}

UNR_IDD = {
    "Normal": 3773, "PortScan": 9498, "TCP-SYN": 9081, "Blackhole": 8420, "Diversion": 5615, "Overflow": 1022
}

DATASETS = [
    ("NSL-KDD", NSL_KDD),
    ("UNSW-NB15", UNSW_NB15),
    ("CICIDS2017", CICIDS2017),
    ("CSE-CIC-IDS2018", CSE_CSC_IDS2018),
    ("ToN_IoT", ToN_IOT),
    ("BoT_IoT", BoT_IoT),
    ("HIKARI-2021", HIKARI_2021),
    ("UNR-IDD", UNR_IDD),
]

# -------------------------
# Run and save results + visuals
# -------------------------
OUTDIR = "cis_results"
os.makedirs(OUTDIR, exist_ok=True)

rows = []
for name, counts in DATASETS:
    rows.append(compute_cis_from_counts(counts, dataset_name=name, beta=0.20))

df = pd.DataFrame(rows).sort_values("CBA_CA", ascending=False).reset_index(drop=True)

csv_path = os.path.join(OUTDIR, "cis_summary.csv")
df.to_csv(csv_path, index=False)
print("Saved:", csv_path)
print(df.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

# Plot: CBA_CA
plt.figure(figsize=(10,5))
x = np.arange(len(df))
plt.bar(x, df["CBA_CA"])
plt.xticks(x, df["dataset"], rotation=30, ha="right")
plt.ylabel("CBA_CA (1 - CIS_CA)")
plt.title("Context-Aware Class Balance Adequacy")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "CBA_CA_bar.png"), dpi=200)
plt.close()
print("Saved:", os.path.join(OUTDIR, "CBA_CA_bar.png"))

# Plot: JS_to_profile
plt.figure(figsize=(10,5))
plt.bar(x, df["JS_to_profile"])
plt.xticks(x, df["dataset"], rotation=30, ha="right")
plt.ylabel("JS Distance to Profile Prior")
plt.title("Deviation from Expected Priors")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "JS_to_profile_bar.png"), dpi=200)
plt.close()
print("Saved:", os.path.join(OUTDIR, "JS_to_profile_bar.png"))

# Heatmap
metrics = ["CIS_all", "CIS_attack", "ABI", "JS_to_profile", "CIS_CA"]
M = df[metrics].values
plt.figure(figsize=(10,5))
plt.imshow(M, aspect="auto")
plt.colorbar()
plt.yticks(np.arange(len(df)), df["dataset"])
plt.xticks(np.arange(len(metrics)), metrics, rotation=30)
plt.title("CIS Components Heatmap")
plt.tight_layout()
plt.savefig(os.path.join(OUTDIR, "CIS_components_heatmap.png"), dpi=200)
plt.close()
print("Saved:", os.path.join(OUTDIR, "CIS_components_heatmap.png"))

print("All done — outputs in folder:", OUTDIR)


Saved: cis_results\cis_summary.csv
        dataset  K  p_benign  CIS_all  CIS_attack       ABI  CIS_base  JS_to_profile   CIS_CA   CBA_CA
        ToN_IoT 10  0.228457 0.030793    0.017725  0.142730  0.049260       0.528422 0.145092 0.854908
        UNR-IDD  6  0.100858 0.048999    0.055098 -0.078970  0.025235       0.735686 0.167325 0.832675
      UNSW-NB15 10  0.360923 0.142640    0.145248  0.289914  0.172877       0.408744 0.220050 0.779950
        NSL-KDD 40  0.518823 0.351270    0.409468  0.506485  0.399772       0.227018 0.365221 0.634779
        BoT_IoT  5  0.000130 0.348483    0.305215 -0.249837  0.215838       0.971857 0.367042 0.632958
    HIKARI-2021  6  0.932113 0.384867    0.247336  0.918536  0.450341       0.045159 0.369305 0.630695
CSE-CIC-IDS2018 15  0.830700 0.672877    0.096760  0.818607  0.529188       0.086485 0.440647 0.559353
     CICIDS2017 15  0.803004 0.632190    0.253176  0.788932  0.549834       0.085365 0.456940 0.543060
Saved: cis_results\CBA_CA_bar.png
Save